<a id="sir"></a>

$$
\text{SIR 传染病模型 (Susceptible-Infected-Recovered Model, 仓室模型 Compartmental Model)} \\
\text{把总人口 N 分成三类, 用比例表示: } s(t) = \frac{S}{N}, \ i(t) = \frac{I}{N}, \ r(t) = \frac{R}{N}, \quad s + i + r = 1 \\
s \text{: 易感者 (Susceptible), } i \text{: 感染者 (Infected), } r \text{: 移出者 (Recovered, 康复或死亡, 不会再被感染)} \\ \quad \\
\text{推导 (从 } \Delta t \text{ 时间内的变化量出发):} \\
\text{1. 感染: 每个感染者单位时间内有效接触 } \beta \text{ 个人, 其中比例 s 是易感者 (质量作用定律 Mass Action)} \\
\Delta t \text{ 内新感染人数 } = \underbrace{I}_{\text{感染者数}} \cdot \underbrace{\beta \Delta t}_{\text{每人接触数}} \cdot \underbrace{s}_{\text{易感比例}} \implies \text{比例为 } \beta s i \, \Delta t \\
\text{2. 康复: 每个感染者单位时间内以概率 } \gamma \text{ 康复, } \Delta t \text{ 内康复比例为 } \gamma i \, \Delta t \\
\text{(康复时间服从参数为 } \gamma \text{ 的指数分布, 平均患病时间 } \frac{1}{\gamma} \text{)} \\
s(t + \Delta t) - s(t) = -\beta s i \Delta t \\
i(t + \Delta t) - i(t) = \beta s i \Delta t - \gamma i \Delta t \\
r(t + \Delta t) - r(t) = \gamma i \Delta t \\
\text{除以 } \Delta t \text{ 令其趋于 0:} \\
\frac{ds}{dt} = -\beta s i, \quad \frac{di}{dt} = \beta s i - \gamma i, \quad \frac{dr}{dt} = \gamma i \\
\text{三式相加: } \frac{d(s + i + r)}{dt} = 0 \text{, 总人口守恒}
$$

数学基础: [从导数建立 ODE](../math_basics/calculus.ipynb#ode) · [指数分布 (平均等待时间)](../math_basics/statistics.ipynb#exponential) · [ODE 方程组](../math_basics/calculus.ipynb#ode-system)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def sir_rhs(t, y, beta, gamma):
    s, i, r = y
    return np.array([-beta * s * i, beta * s * i - gamma * i, gamma * i])

def rk4(f, t0, y0, h, n_steps, *args):
    y = np.array(y0, dtype=float)
    ts, ys = [t0], [y.copy()]
    t = t0
    for _ in range(n_steps):
        k1 = f(t, y, *args)
        k2 = f(t + h / 2, y + h / 2 * k1, *args)
        k3 = f(t + h / 2, y + h / 2 * k2, *args)
        k4 = f(t + h, y + h * k3, *args)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t += h
        ts.append(t); ys.append(y.copy())
    return np.array(ts), np.array(ys)

beta, gamma = 0.3, 0.1          # 平均患病 10 天
y0 = [0.999, 0.001, 0.0]
h, n_steps = 0.1, 2000
ts, ys = rk4(sir_rhs, 0.0, y0, h, n_steps, beta, gamma)
sol = solve_ivp(sir_rhs, [0, ts[-1]], y0, t_eval=ts, args=(beta, gamma), rtol=1e-10, atol=1e-12)
print(f'手写 RK4 == solve_ivp ? {np.allclose(ys.T, sol.y, atol=1e-7)}')
print(f's + i + r 的范围 = [{ys.sum(axis=1).min():.10f}, {ys.sum(axis=1).max():.10f}]')

# 欧拉法 (直接用差分方程) 作对比: 步长大时误差明显
ys_euler = [np.array(y0)]
for _ in range(200):
    ys_euler.append(ys_euler[-1] + 1.0 * sir_rhs(0, ys_euler[-1], beta, gamma))
ys_euler = np.array(ys_euler)

plt.figure(figsize=(4, 3))
for k, name in enumerate(['s', 'i', 'r']):
    plt.plot(ts, ys[:, k], label=f'{name} (RK4)')
plt.plot(np.arange(201), ys_euler[:, 1], 'k:', label='i (Euler, h=1)')
plt.xlabel('day')
plot_nicely()
plt.show()

<a id="r0"></a>

$$
\text{基本再生数 (Basic Reproduction Number) } R_0 = \frac{\beta}{\gamma} \\
\text{含义: 在所有人都是易感者时, 一个感染者在患病期间平均传染的人数 = 每天传染 } \beta \text{ 人} \times \text{患病 } \frac{1}{\gamma} \text{ 天} \\ \quad \\
\text{阈值定理 (Threshold Theorem):} \\
\frac{di}{dt} = i (\beta s - \gamma) > 0 \iff s > \frac{\gamma}{\beta} = \frac{1}{R_0} \\
\text{1. 初始时 } s_0 \approx 1 \text{: } R_0 > 1 \text{ 疫情爆发, } R_0 < 1 \text{ 感染人数一开始就下降} \\
\text{2. s 单调递减 (} \frac{ds}{dt} < 0 \text{), i 在 } s = \frac{1}{R_0} \text{ 时达到峰值, 之后下降} \\
\text{3. 群体免疫阈值 (Herd Immunity Threshold): 接种疫苗让初始易感比例 } s_0 < \frac{1}{R_0} \text{, 即免疫比例 } > 1 - \frac{1}{R_0}
$$

In [ ]:
R0 = beta / gamma
i_curve = ys[:, 1]
k_peak = np.argmax(i_curve)
print(f'R0 = {R0:.2f}, 峰值时 s = {ys[k_peak, 0]:.4f}, 1 / R0 = {1 / R0:.4f}, 峰值时间 = 第 {ts[k_peak]:.1f} 天')
print(f'群体免疫阈值 = 1 - 1/R0 = {1 - 1 / R0:.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for R0_try in [0.8, 1.5, 3.0, 5.0]:
    _, y_try = rk4(sir_rhs, 0.0, y0, 0.2, 1500, R0_try * gamma, gamma)
    axs[0].plot(np.arange(1501) * 0.2, y_try[:, 1], label=f'R0 = {R0_try}')
axs[0].set_xlabel('day')
axs[0].set_ylabel('i(t)')

# 疫苗: 免疫比例 v 的人直接进入 R
for v in [0.0, 0.4, 0.6, 0.7]:
    _, y_v = rk4(sir_rhs, 0.0, [1 - v - 0.001, 0.001, v], 0.2, 1500, beta, gamma)
    axs[1].plot(np.arange(1501) * 0.2, y_v[:, 1], label=f'vaccinated {v:.0%}')
axs[1].set_xlabel('day')
plot_nicely()
plt.show()

<a id="sir-conserved"></a>

$$
\text{守恒量与最终规模 (Final Size)} \\ \quad \\
\text{消去时间 t: 用链式法则 } \frac{di}{ds} = \frac{di/dt}{ds/dt} = \frac{\beta s i - \gamma i}{-\beta s i} = -1 + \frac{\gamma}{\beta s} = -1 + \frac{1}{R_0 s} \\
\text{两边对 s 积分: } i = -s + \frac{1}{R_0} \ln s + C \\
i + s - \frac{1}{R_0} \ln s = \text{常数} = i_0 + s_0 - \frac{1}{R_0} \ln s_0 \quad (\text{相平面 (s, i) 上的轨线}) \\ \quad \\
\text{1. 峰值: 在 } s = \frac{1}{R_0} \text{ 处} \\
i_{\max} = i_0 + s_0 - \frac{1}{R_0} \ln s_0 - \frac{1}{R_0} + \frac{1}{R_0} \ln \frac{1}{R_0} = i_0 + s_0 - \frac{1}{R_0}\left(1 + \ln(R_0 s_0)\right) \\ \quad \\
\text{2. 最终规模: } t \to \infty \text{ 时 } i \to 0 \text{, 设 } s_0 \approx 1, i_0 \approx 0 \text{, 最终感染比例 } r_\infty = 1 - s_\infty \\
s_\infty - \frac{1}{R_0} \ln s_\infty = 1 \implies \ln s_\infty = -R_0 (1 - s_\infty) = -R_0 r_\infty \\
r_\infty = 1 - e^{-R_0 r_\infty} \quad (\text{超越方程, 没有解析解}) \\ \quad \\
\text{3. 用牛顿法解: } g(r) = r - 1 + e^{-R_0 r} = 0, \quad g'(r) = 1 - R_0 e^{-R_0 r} \\
r_{k+1} = r_k - \frac{g(r_k)}{g'(r_k)} \quad (r = 0 \text{ 永远是一个根, 要从 } r_0 = 1 \text{ 附近开始迭代找正根}) \\
\text{结论: 疫情结束时并不是所有人都被感染过, 有 } s_\infty > 0 \text{ 的人从未感染 (感染者太少, 传不下去)}
$$

数学基础: [链式法则](../math_basics/calculus.ipynb#derivative-rules) · [牛顿法](../math_basics/calculus.ipynb#newton)

In [ ]:
conserved = ys[:, 1] + ys[:, 0] - np.log(ys[:, 0]) / R0
print(f'守恒量 i + s - ln(s)/R0 的变化范围 = {np.ptp(conserved):.2e} (数值上恒定)')

s0, i0 = y0[0], y0[1]
i_max_theory = i0 + s0 - (1 + np.log(R0 * s0)) / R0
print(f'峰值: 理论 = {i_max_theory:.6f}, 模拟 = {i_curve.max():.6f}')

def final_size_newton(R0, r_init=1.0, tol=1e-14, max_iter=50):
    r = r_init
    for k in range(max_iter):
        g = r - 1 + np.exp(-R0 * r)
        dg = 1 - R0 * np.exp(-R0 * r)
        r_new = r - g / dg
        if abs(r_new - r) < tol:
            return r_new, k + 1
        r = r_new
    return r, max_iter

r_inf, iters = final_size_newton(R0)
r_brent = brentq(lambda r: r - 1 + np.exp(-R0 * r), 1e-6, 1)
print(f'最终规模: 牛顿法 = {r_inf:.10f} ({iters} 次迭代), brentq = {r_brent:.10f}, 模拟 (第 200 天) r = {ys[-1, 2]:.6f}')
print(f'从未感染的比例 s_inf = {1 - r_inf:.4f}')

R0_grid = np.linspace(0.5, 5, 100)
final_sizes = [final_size_newton(R)[0] if R > 1 else 0.0 for R in R0_grid]
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(R0_grid, final_sizes, label='r_inf from Newton')
axs[0].plot(R0_grid, np.maximum(0, 1 - 1 / R0_grid), '--', label='herd immunity 1 - 1/R0')
axs[0].set_xlabel('R0')
# 相平面: 不同初值的轨线
for i_start in [0.001, 0.1, 0.3, 0.5]:
    _, y_ph = rk4(sir_rhs, 0.0, [1 - i_start, i_start, 0], 0.2, 1500, beta, gamma)
    axs[1].plot(y_ph[:, 0], y_ph[:, 1], label=f'i0 = {i_start}')
axs[1].axvline(1 / R0, color='gray', linestyle=':', label='s = 1/R0 (peak)')
axs[1].set_xlabel('s')
axs[1].set_ylabel('i')
plot_nicely()
plt.show()

<a id="seir"></a>

$$
\text{SEIR 模型: 加入潜伏者 (Exposed), 被感染后要经过平均 } \frac{1}{\sigma} \text{ 天的潜伏期才有传染性} \\
\frac{ds}{dt} = -\beta s i, \quad \frac{de}{dt} = \beta s i - \sigma e, \quad \frac{di}{dt} = \sigma e - \gamma i, \quad \frac{dr}{dt} = \gamma i \\
\text{推导方式完全相同, 只是 "感染" 先流入 E, 再以速率 } \sigma \text{ 流入 I} \\
\text{潜伏者最终都会变成感染者, 所以 } R_0 = \frac{\beta}{\gamma} \text{ 不变, 最终规模也相同, 但疫情来得更慢, 峰值更低更晚} \\
\text{(其他变种: SIS 康复后重新易感, SIRS 免疫会消失, 加入出生死亡等)}
$$

In [ ]:
def seir_rhs(t, y, beta, sigma, gamma):
    s, e, i, r = y
    return np.array([-beta * s * i, beta * s * i - sigma * e, sigma * e - gamma * i, gamma * i])

sigma = 0.2     # 平均潜伏 5 天
t_eval = np.linspace(0, 400, 2001)
sol_seir = solve_ivp(seir_rhs, [0, 400], [0.999, 0, 0.001, 0], t_eval=t_eval, args=(beta, sigma, gamma), rtol=1e-10, atol=1e-12)
sol_sir = solve_ivp(sir_rhs, [0, 400], [0.999, 0.001, 0], t_eval=t_eval, args=(beta, gamma), rtol=1e-10, atol=1e-12)
_, ys_seir_rk4 = rk4(seir_rhs, 0.0, [0.999, 0, 0.001, 0], 0.2, 2000, beta, sigma, gamma)
print(f'SEIR 手写 RK4 == solve_ivp ? {np.allclose(ys_seir_rk4.T, sol_seir.y, atol=1e-7)}')
print(f'SIR : 峰值 i = {sol_sir.y[1].max():.4f}, 第 {t_eval[np.argmax(sol_sir.y[1])]:.0f} 天, 最终 r = {sol_sir.y[2, -1]:.4f}')
print(f'SEIR: 峰值 i = {sol_seir.y[2].max():.4f}, 第 {t_eval[np.argmax(sol_seir.y[2])]:.0f} 天, 最终 r = {sol_seir.y[3, -1]:.4f}')
print(f'最终规模理论值 (两者相同) = {final_size_newton(R0, r_init=1.0)[0]:.4f} (初值 s0 = 0.999 时略有差别)')

plt.figure(figsize=(4, 3))
plt.plot(t_eval, sol_sir.y[1], label='SIR  i(t)')
plt.plot(t_eval, sol_seir.y[2], label='SEIR i(t)')
plt.plot(t_eval, sol_seir.y[1], '--', label='SEIR e(t)')
plt.xlabel('day')
plot_nicely()
plt.show()